<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Bonus (optionnel, avancé) — Autoencodeurs parcimonieux & signal émergent
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Rappel : entraîner un autoencodeur dense et voir pourquoi son goulot d'étranglement reste polysémantique<br>
    - Entraîner un autoencodeur parcimonieux (Top-K) sur les activations Evo2 pour obtenir des caractéristiques isolées<br>
    - Chercher une structure émergente sans aucune étiquette — par ex. une périodicité de 3 liée au cadre de lecture<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Cette piste n'est **pas obligatoire** pour le livrable principal — elle est là si votre
groupe termine en avance et veut voir ce que l'apprentissage de représentation non
supervisé peut découvrir, avec **zéro** étiquette, à l'intérieur d'un modèle de langage
génomique.

#### **Partie 1 — Rappel : l'autoencodeur classique**

Un autoencodeur apprend à compresser son entrée à travers un **goulot d'étranglement**
(bottleneck) étroit et à la reconstruire — ce goulot le force à ne garder que la structure
la plus utile. Voyons cela d'abord sur nos propres vecteurs de k-mers, car c'est rapide et
ne nécessite rien d'Evo2.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day5/assets/AE.png"/>

In [ ]:
import sys
sys.path.append("src")

import torch
import torch.nn as nn
from data import load_all
from featurize import kmer_matrix

train_df = load_all("../../data/supervised/processed", max_rows=None)["train"]
X = torch.tensor(kmer_matrix(train_df["sequence"].tolist(), k=4), dtype=torch.float32)

d_in = X.shape[1]
# TODO : construisez un autoencodeur dense avec un goulot d'étranglement à 16 dimensions
# (Linear(d_in, 16) -> ReLU -> Linear(16, d_in))
vanilla_ae = ...
optimizer = ...

for epoch in range(20):
    optimizer.zero_grad()
    # TODO : reconstruction + perte MSE entre x_hat et X
    x_hat = ...
    loss = ...
    loss.backward()
    optimizer.step()
print(f"final reconstruction loss: {loss.item():.5f}")

Ce goulot d'étranglement est **dense** : chacun des 16 nombres est utilisé
pour chaque entrée, et ils ne sont pas individuellement interprétables — une seule
« caractéristique » est généralement un mélange de nombreuses causes sous-jacentes (c'est
la *polysémanticité*).

#### **Partie 2 — Ce qu'apporte un autoencodeur parcimonieux**

Un autoencodeur **parcimonieux** (SAE) fait l'inverse : au lieu d'un goulot *étroit*, il
utilise un goulot **surcomplet** (plus d'unités cachées que d'entrées), mais force
seulement une poignée (`k`) d'entre elles à être actives pour une entrée donnée. L'idée
(issue de recherches récentes en interprétabilité mécanistique sur les modèles de
langage) : avec suffisamment de capacité et la bonne parcimonie, les caractéristiques
individuelles ont tendance à devenir **monosémantiques** — chacune suit un motif
distinct, ce qui les rend inspectables.

Ici, nous entraînons un SAE sur des **activations brutes d'Evo2** — des états internes
par position de nucléotide, extraits sans aucune information d'étiquette — et nous
cherchons des caractéristiques qui correspondent à quelque chose de réel, comme le cadre
de lecture d'un codon (structure de période 3).

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day5/assets/SAE.png"/>

In [ ]:
from embeddings import load_autoencoder_activations
from models.sae import TopKSparseAutoencoder, train_sae

acts = load_autoencoder_activations("../2-data/autoencoder", "train")
print("full activation dump shape:", acts.shape)

# on ne charge en mémoire qu'un sous-échantillon — c'est un memmap sur un fichier de 19 Go,
# ne le chargez pas en entier
N_SUBSAMPLE = 20_000
# TODO : convertissez acts[:N_SUBSAMPLE] en tensor float32
X_acts = ...
print("training on:", X_acts.shape)

In [ ]:
# TODO : instanciez TopKSparseAutoencoder(d_in=X_acts.shape[1], d_hidden=X_acts.shape[1] * 8, k=32)
# puis entraînez-le avec train_sae (10 époques, batch_size=512)
sae = ...
sae, history = ...

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Partie 3 — Cherchons une structure émergente : la périodicité**

L'ADN est lu trois nucléotides à la fois (codons). Si le SAE a découvert quelque chose sur
le cadre de lecture sans aucune supervision, certaines caractéristiques devraient
s'activer périodiquement avec une période ~3 le long d'un segment continu de positions.

In [ ]:
import numpy as np
from viz import plot_sae_feature_activity

# un segment continu de positions de tokens consécutives depuis le memmap
window = torch.tensor(acts[1000:1000 + 300].astype("float32"))
with torch.no_grad():
    # TODO : passez `window` dans le sae pour récupérer les caractéristiques (deuxième sortie)
    _, features = ...
features = features.numpy()

# on classe les caractéristiques par variance d'activation (un proxy simple de "fait
# quelque chose d'intéressant")
variances = features.var(axis=0)
top_features = np.argsort(variances)[::-1][:5]

plot_sae_feature_activity(features[:, top_features], top_features)

In [ ]:
# vérification rapide de périodicité via FFT — cherchez un pic proche de la fréquence 1/3
for feat_idx in top_features:
    signal = features[:, feat_idx] - features[:, feat_idx].mean()
    # TODO : calculez le spectre d'amplitude (np.fft.rfft) et les fréquences associées (np.fft.rfftfreq)
    spectrum = ...
    freqs = ...
    period = 1 / freqs[np.argmax(spectrum[1:]) + 1] if len(spectrum) > 1 else float("nan")
    print(f"feature {feat_idx}: dominant period ~ {period:.2f} positions")

#### **Discussion**

- Une caractéristique est-elle ressortie avec une période proche de 3 ? Cela suggérerait
  que le modèle — sans étiquette, sans tâche, juste « reconstruire mes propres
  activations internes » — a représenté le cadre de lecture en interne.
- Essayez de corréler les principales caractéristiques avec les étiquettes
  codant/non-codant de `data/supervised/processed` pour la même région du génome (il
  faudra aligner les positions de tokens avec les coordonnées du génome) — certaines
  caractéristiques séparent-elles les régions codantes des non-codantes toutes seules ?
- C'est exploratoire, non noté sur l'obtention d'un « vrai » résultat — l'important est de
  voir qu'une structure interprétable *peut* émerger d'un entraînement purement non
  supervisé.

*Bloqué ? La version complète est dans
`solution/bonus_sparse_autoencoder_interpretability.ipynb`.*

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin de la semaine 4 &mdash; f&eacute;licitations !</div>
      <div>D&#39;un g&eacute;nome brut &agrave; un classifieur distill&eacute; : donn&eacute;es, mod&egrave;les de r&eacute;f&eacute;rence, embeddings Evo2,
      distillation, compression &mdash; et un aper&ccedil;u de l&#39;interpr&eacute;tabilit&eacute;. Gardez vos notebooks :
      ils sont la trace de ce que vous avez construit.</div>
      <div style="margin-top: 0.5em; font-size: 0.85em;">EEIA &middot; bioAI Workshop &mdash; Semaine 4</div>
    </div>
  </div>
</div>